# 01 — Bronze ingestion
Reads the public Olist CSV extracts from the Lakehouse Files area and writes source-shaped Delta tables. This first version is a reproducible full snapshot load.

In [ ]:
from pyspark.sql import functions as F

SOURCE = "Files/source/olist"

source_tables = {
    "orders": "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
}

def ingest_csv(table_name: str, file_name: str):
    path = f"{SOURCE}/{file_name}"
    df = (spark.read
          .option("header", True)
          .option("inferSchema", True)
          .option("mode", "PERMISSIVE")
          .csv(path))
    df = (df
          .withColumn("source_file_name", F.lit(file_name))
          .withColumn("ingested_at", F.current_timestamp()))
    (df.write
       .format("delta")
       .mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(f"bronze.{table_name}"))
    return df.count()

counts = {}
for table_name, file_name in source_tables.items():
    counts[table_name] = ingest_csv(table_name, file_name)

print("Bronze ingestion completed:")
display(spark.createDataFrame([(k, int(v)) for k, v in counts.items()], ["table_name", "rows_loaded"]))

## Bronze validation
The following checks confirm that the expected tables were created and contain records.

In [ ]:
for table_name in source_tables:
    print(table_name, spark.table(f"bronze.{table_name}").count())